# Training dataset of the surrogates

Downloads `equilibrium_a_3k_spline_input` from Hugging Face into this folder, unless it is already here, and shows what one row holds. `README.md` describes the columns and how the equilibria were generated.

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    desc_truth_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    plot as ideal_mhd_plot,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
    download_from_hugging_face,
)

DATA_DIR = pathlib.Path(download_from_hugging_face.__file__).parents[3] / "data"
DATASET = dataset_utils.DEFAULT_DATASET
PARQUET = DATA_DIR / f"{DATASET}.parquet"
SETTINGS = DATA_DIR / f"{DATASET}_settings.json"
for path in (PARQUET, SETTINGS):
    download_from_hugging_face.download(
        download_from_hugging_face.DATASETS_REPOSITORY, path.name, path
    )


One row per DESC equilibrium. The settings JSON holds the spectral basis of the `R_lmn`, `Z_lmn` and `lambda_lmn` columns and the train/val/test split.

In [ ]:
equilibria = pd.read_parquet(PARQUET)
metadata = dataset_utils.load_metadata(SETTINGS)
print(equilibria.shape)
print(metadata.basis.resolution)
print(
    "split:",
    len(metadata.train_ids),
    len(metadata.validation_ids),
    len(metadata.test_ids),
)
equilibria.head(3)

## Dataset visualizations
- Distribution of the volume-averaged normalized force residual over all equilibria.
- Pressure profiles of the first ten equilibria, the input of the surrogates.
- Nested flux surfaces of the first equilibrium in six toroidal sections.

In [ ]:
figure, axis = plt.subplots()
axis.hist(equilibria["fnorm_volume_averaged_normalized_force_error"], bins=50)
axis.set_xlabel("normalized force residual")
axis.set_ylabel("equilibria")
plt.show()

In [ ]:
figure, axis = plt.subplots()
for row in equilibria.head(10).itertuples():
    axis.plot(row.pressure_knots, row.pressure_params / 1e3)
axis.set_xlabel("rho")
axis.set_ylabel("pressure [kPa]")
axis.set_title("Pressure profiles of the first ten equilibria")
plt.show()

In [ ]:
truth = desc_truth_types.read_desc_truth(
    equilibria["desc_equilibrium_id"].iloc[0], PARQUET, SETTINGS, machine_scale=None
)
figure, axes = ideal_mhd_plot.plot_surfaces(truth)
plt.show()
